# 镜像中转搬运 notebook（本地 Docker Hub → 华为云 SWR cn-north-4）

> **何时需要**：仅当你的机器在**国外/国际线路**、要把镜像传进**国内** SWR（如北京 cn-north-4，
> 国际入中上行被限 ~20KB/s，docker push 全量层基本传不动）才走这条中转路。
> 机器在中国大陆的话**不需要本 notebook**：`docker login` + `docker push` 直推即可（README.md 阶段 4A 主路线）。
>
> 原理：本机先把镜像推到 Docker Hub（国际线路对国际枢纽快），再在**同区域的
> ModelArts CPU notebook** 里用 crane 把镜像从 Docker Hub 搬到 SWR（notebook 与
> SWR 同在华为云内网，实测双向 ~8MB/s 量级）。
>
> **跑法**：替换占位符后**按 cell 顺序**执行——① 下载 crane → ② 登录两端 →
> ③ 镜像源探测 → ④ 搬运；每个 cell 等跑完再下一个。
> 占位符：`<DOCKER_HUB_USER>` / `<DOCKER_HUB_TOKEN>` / `<SWR_LOGIN_USER>` / `<SWR_LOGIN_PASSWORD>` /
> `<IMAGE>` / `<TAG>` / `<SWR_ORG>`。登录指令在 SWR 控制台「客户端上传 → 登录指令」里整条复制。

**坑位速查**（全部实证，2026-09-01 实跑复盘）：
- **crane 下载截断**（GitHub 直连被掐）：`tar` 解压会留下**半截二进制**——搬运 cell 表现为
  退出码 `-11`（SIGSEGV 秒崩、日志 0 字节），下载 cell 表现为 `tar: Unexpected EOF`；且截断的
  curl 可能以退出码 0 结束，`||` 兜底不触发。→ 下载 cell 已内置：代理前缀探测 → 断点续传
  （不支持 Range 就整档重下）→ `tar tzf` 通读校验通过才解压，解压前先删残缺文件。
  代理前缀随时失效（ghfast.top / gh-proxy.com / ghproxy.net …换着试）；
- **SRC 镜像源会挂**：dockerproxy.net 实测间歇 502 Bad Gateway → 换源即可（docker.1ms.run
  当次稳）。copy 前跑「镜像源探测」cell 挑活源；manifest 探测通过同时证明仓库已 public、
  tag 已在 Docker Hub。换源重跑不亏：crane 跳过远端已有层与 manifest（幂等，重复运行秒级
  打 `existing manifest`——那行是"远端已有这份内容"，**不是"没传"**）；
- **notebook 停止/重启会清空 `/tmp`**：crane、日志全没了、kernel 变量也没了——表现为 cell
  毫秒级"跑完"（如 939 ms）但实际抛了 FileNotFoundError / NameError，traceback 就在 cell
  下方（最常被忽略的地方）。→ 从 cell ① 起重跑完整流程；
- **拉取代理是匿名的，只服务 public 仓库**：Docker Hub 仓库须设 public；登录 cell 对
  docker.io 的凭证对代理无效（crane 凭证按 registry 主机隔离）。DaoCloud 源
  （docker.m.daocloud.io）有白名单，拒个人仓库，别用；
- **copy 期间 SWR 上看不到任何东西是正常的**：manifest 最后才推；verbose 全写进日志文件
  不进 cell。搬运 cell 结束时打的 ✅/❌ 横幅才是结论；
- 必须加 `--platform linux/amd64` 剥掉 buildx attestation index，否则 SWR 报 `MANIFEST_INVALID`
  （与本地构建带 `--provenance=false` 是同一个坑的两端）。

**验收**：✅ 横幅里的 digest == SWR 控制台版本详情里的 digest == 本机
`docker image inspect <镜像>` 的 `.Descriptor.digest`，三方一致才算搬完。
SWR 控制台右上角区域切到**华北-北京四**，镜像挂在**组织**下看。


In [ ]:
import os, subprocess

os.makedirs("/tmp/relay", exist_ok=True)
%cd /tmp/relay

# GitHub 直连可能截断（截断的 tar 会解压出残缺 crane，一跑就段错误 -11）——
# 先探测哪个代理前缀活着且返回真文件，再下载；支持 Range 就断点续传，不支持就整档重下。
GH = "https://github.com/google/go-containerregistry/releases/download/v0.20.3/go-containerregistry_Linux_x86_64.tar.gz"
CANDIDATES = ["https://ghfast.top/", "https://gh-proxy.com/", "https://ghproxy.net/"]

# 清掉可能的残缺压缩包 / 半截二进制（上次截断的遗产）
!rm -f crane.tar.gz crane

def tarball_ok():
    return subprocess.run(["tar", "tzf", "crane.tar.gz"], capture_output=True).returncode == 0

good = []
for p in CANDIDATES:
    r = subprocess.run(["curl", "-sIL", "-m", "15", p + GH], capture_output=True, text=True)
    ls = r.stdout.lower().splitlines()
    code = next((l for l in reversed(ls) if l.startswith("http")), "")   # 取重定向后的最终状态行
    cl = [int(l.split(":", 1)[1]) for l in ls
          if l.startswith("content-length") and l.split(":", 1)[1].strip().isdigit()]
    print(f"{p:28s} {code:24s} {cl}")
    if "200" in code and cl and cl[-1] > 5_000_000:   # 活着且像真文件（>5MB），不是报错页
        good.append((cl[-1], p))

assert good, "三个代理都不通——换一批前缀重跑，或改走本机下载→传OBS→moxing取回"
expect, prefix = max(good)
url = prefix + GH
print(f"选定 {url}  期望 {expect/1e6:.1f} MB")

for i in range(15):
    subprocess.run(["curl", "-sL", "-C", "-", "-o", "crane.tar.gz", url])   # 支持 Range 就续传
    if os.path.getsize("crane.tar.gz") < expect:
        subprocess.run(["curl", "-sL", "-o", "crane.tar.gz", url])          # 不支持 Range 整档重下
    size = os.path.getsize("crane.tar.gz")
    print(f"第 {i+1} 轮：{size/1e6:.1f} / {expect/1e6:.1f} MB")
    if size >= expect:
        break
assert tarball_ok(), "gzip CRC 没过——再跑一轮本 cell"
!tar xzf crane.tar.gz crane
!./crane version


In [ ]:
# 登录两端 registry（密码走命令行参数，notebook 用完即弃；Docker Hub 用 access token 别用主密码）
# SWR 登录指令：SWR 控制台右上「客户端上传 → 登录指令」，user 形如 'cn-north-4@<账号域名>'
!./crane auth login docker.io -u <DOCKER_HUB_USER> -p <DOCKER_HUB_TOKEN>
!./crane auth login swr.cn-north-4.myhuaweicloud.com -u <SWR_LOGIN_USER> -p <SWR_LOGIN_PASSWORD>

In [ ]:
# 镜像源探测（copy 前跑）：SRC 走的是 Docker Hub 国内拉取代理，代理随时会死
# （dockerproxy.net 实测间歇 502）。逐个试 manifest——小请求秒级出结果，OK 的都能当 SRC；
# 能拉到 manifest 同时也证明：仓库已 public、tag 确实在 Docker Hub 上。
import subprocess

DOCKER_HUB_USER = "<DOCKER_HUB_USER>"    # 与 copy cell 里 SRC_FMT 同一个
IMAGE, TAG = "<IMAGE>", "<TAG>"          # 先探测要搬的那个 tag

MIRRORS = ["docker.1ms.run", "docker.1panel.live", "dockerproxy.net",
           "hub.rat.dev", "dockerpull.org", "docker.chenby.cn"]

alive = []
for m in MIRRORS:
    ref = f"{m}/{DOCKER_HUB_USER}/{IMAGE}:{TAG}"
    try:
        r = subprocess.run(["./crane", "manifest", "--platform", "linux/amd64", ref],
                           capture_output=True, text=True, timeout=60)
        ok = r.returncode == 0
        print(f"{m:22s}", "OK" if ok else (r.stderr.strip().splitlines() or ["fail"])[-1][:70])
        if ok:
            alive.append(m)
    except subprocess.TimeoutExpired:
        print(f"{m:22s} 超时")
print("可用源:", alive or "无——换时间段再试，或走兜底：本机 docker save → 传 OBS → moxing 取回 → crane push")


In [ ]:
import os, re, subprocess, time

# 要搬的 tag（可一次列多个）；SRC 用镜像源探测 cell 挑出的活源（502 就换，重跑幂等）。
TAGS = ["<TAG>"]
SRC_FMT = "docker.1ms.run/<DOCKER_HUB_USER>/<IMAGE>:{t}"
DST_FMT = "swr.cn-north-4.myhuaweicloud.com/<SWR_ORG>/<IMAGE>:{t}"

LOG = "/tmp/relay/crane_copy.log"
BAR = "=" * 62

def new_log_lines(offset):
    """只取本次运行追加进日志的部分，避免混入历史运行的旧输出。"""
    with open(LOG, "r", errors="replace") as f:
        f.seek(offset)
        return f.read().splitlines()

for TAG in TAGS:
    src, dst = SRC_FMT.format(t=TAG), DST_FMT.format(t=TAG)
    offset = os.path.getsize(LOG) if os.path.exists(LOG) else 0
    print(f"▶ 开始搬运 {src} -> {dst}")
    proc = subprocess.Popen(
        ["./crane", "copy", "-v", "--platform", "linux/amd64", src, dst],
        stdout=open(LOG, "a"), stderr=subprocess.STDOUT, cwd="/tmp/relay")
    t0 = time.time()
    rc = proc.wait()
    dt = time.time() - t0
    lines = new_log_lines(offset)

    whole = chr(10).join(lines)
    digests = re.findall(r"(?:existing|pushed) manifest.*?(sha256:[0-9a-f]{64})", whole)
    errors = [l for l in lines
              if l.startswith("Error") or "unexpected status code" in l]

    if rc == 0:
        mode = ("远端原本就有（existing，未重复传）"
                if any("existing manifest" in l for l in lines)
                else "本次推送完成（pushed）")
        print(f"""
{BAR}
✅✅✅ 成功   tag={TAG}   耗时 {dt:.0f}s
    digest : {digests[-1] if digests else '(未解析到，看日志确认)'}
    说明   : {mode}
    下一步 : SWR 控制台（华北-北京四 → 你的组织）可见；阶段 5 建作业直接选它
{BAR}""")
    else:
        why = (f"进程被信号 {-rc} 杀死（-11=SIGSEGV：crane 二进制损坏，重跑下载 cell）"
               if rc < 0 else "crane 报错退出")
        print(f"""
{BAR}
❌❌❌ 失败   tag={TAG}   退出码 {rc}   耗时 {dt:.0f}s
    原因   : {why}
    错误摘录（仅本次运行）:""")
        for l in (errors or [l for l in lines if l.strip()][-3:])[-5:]:
            print("      " + l[:150])
        print(f"""    处置   : 502/超时 → 换镜像源探测 cell 里的活源重跑（幂等，已传层不重传）；
             unauthorized → Docker Hub 仓库设 public；
             MANIFEST_UNKNOWN → tag 没推上 Docker Hub，先本机 docker push
    完整日志: {LOG}
{BAR}""")

# 验收：SWR 控制台（华北-北京四 → 组织）看到镜像，且 digest 与本机
#   docker image inspect <镜像> --format 的 .Descriptor.digest 一致


## 排障速查（按现象对号）

1. **cell "跑完"只要几百毫秒、没输出** → 根本没跑起来：notebook 重启过（/tmp 清空、
   变量没了），traceback 就在 cell 下方。从 cell ① 重跑。诊断三连（新 cell 里跑）：

   ```
   !ls -la /tmp/relay/
   !cd /tmp/relay && ./crane version
   !tail -n 30 /tmp/relay/crane_copy.log
   ```

2. **❌ 退出码 -11（秒崩、日志 0 字节）** → crane 二进制残缺（下载截断的遗产），重跑 cell ①；
3. **❌ 502 Bad Gateway / 超时** → SRC 镜像源挂了：跑 cell ③ 换活源，重跑 cell ④（幂等不重传）；
4. **❌ unauthorized** → Docker Hub 仓库没设 public；
   **❌ MANIFEST_UNKNOWN** → tag 没推上 Docker Hub，先本机 `docker push docker.io/<用户>/<镜像>:<tag>`；
5. **✅ 了但 SWR 控制台看不到** → 右上角区域切**华北-北京四**；左侧**组织管理**进你的
   组织看（镜像挂组织下，总览页没有）；
6. **cell ④ 长时间没动静** → 正常，进度在 `/tmp/relay/crane_copy.log`（可 `!tail -f` 跟踪），
   401MB 量级要几分钟。

## 兜底路线：本机 tar 直传（完全绕开 Docker Hub，通道全已验证）

镜像本来就在本机时最稳的路——不依赖 Docker Hub、不依赖拉取代理：

```powershell
# 本机（PowerShell）：先确认纯 manifest（是 index 就带 --provenance=false 重建），再导出
docker image inspect <镜像> --format "{{json .Descriptor}}"
docker save <镜像> -o dpo-image.tar
# 控制台上传到 obs://<桶>/tools/dpo-image.tar（OBS 控制台「上传对象」直传；别放 code-dir/）
```

```python
# notebook：
import moxing as mox
mox.file.copy("obs://<桶>/tools/dpo-image.tar", "/tmp/relay/image.tar")
%cd /tmp/relay
!./crane push image.tar swr.cn-north-4.myhuaweicloud.com/<SWR_ORG>/<IMAGE>:<TAG>
```

## 收尾

- 中转完成 → **停掉 notebook**（按小时计费）；
- Docker Hub 上的中转仓库可转回 private 或删除（镜像已进 SWR，中转使命完成）；
- 用本地新版 notebook 覆盖云端旧版时，占位符记得重新替换（登录凭证一次性，覆盖无泄露风险）。
